# Sección 0 · Introducción metodológica

*El flujo de análisis del curso: del archivo de datos a la conclusión*

> **Esta sección no tiene ejercicios:** presenta la metodología del curso. No entraremos en la biología de la especie (es tu tema): nos centramos en **cómo** analizar datos SNP, qué asume cada paso y cómo justificar cada decisión.


## Objetivos

Al terminar esta sección sabrás:

1. El **flujo de análisis** del curso: del archivo VCF a la conclusión.
2. De dónde sale un VCF y **qué contiene**.
3. Qué datos usaremos (simulados vs. reales) y por qué.


## 1. El flujo del curso

El curso recorre un análisis reproducible en cuatro grandes bloques:

```
1. Importar y limpiar  →  VCF → matriz de dosis → control de calidad (QC)
2. Diversidad          →  Ho, He, Fis → ¿hay diferencias entre poblaciones?
3. Estructura          →  Fst, PCA, ancestría → ¿cómo se agrupan los individuos?
4. Ambiente            →  RDA → ¿cuánta variación se asocia al clima?
```

**El hilo conductor:** en cada paso hay decisiones (umbrales, modelos, parámetros) que cambian el resultado. Aprenderemos a tomarlas **justificadas** y a **reportarlas**. Un número sin incertidumbre ni sesgo conocido no es un resultado.


## 2. De dónde sale un VCF: el pipeline

Un VCF no aparece de la nada: es el final de una **tubería bioinformática** (*pipeline*). El camino completo:

```
1. Muestreo      → se colecta tejido de cada individuo, anotando sitio y coordenadas.
2. Extracción    → se aísla el ADN.
3. Secuenciación → un secuenciador (Illumina) lee fragmentos cortos de ADN (reads).
4. Mapeo         → cada read se alinea a un genoma de referencia.
5. Llamado       → el software (GATK HaplotypeCaller) detecta variantes: posiciones donde
                   una muestra difiere de la referencia.
6. Filtrado      → se descartan variantes de baja calidad (poca cobertura, sesgo de hebra…).
7. VCF           → el resultado: una variante por fila, un genotipo por muestra.
```

Cada paso puede introducir **errores y sesgos**. Buena parte del curso consiste en diagnosticarlos y mitigarlos (eso es el *control de calidad*).

**Una variante (SNP)** es una posición del genoma donde individuos distintos llevan nucleótidos distintos. El genotipo se codifica como `0/0`, `0/1`, `1/1` o `./.` (faltante), y lo convertimos a **dosis** (`0`, `1`, `2`, `NA`) para poder hacer aritmética.


## 3. El formato VCF

- **Líneas `##`**: metadatos (formato, definición de campos).
- **Línea `#CHROM`**: las columnas — 9 fijas + una por individuo.
- **Filas**: una por variante.

| Columna | Qué es |
|---|---|
| `CHROM`, `POS` | contig y posición de la variante |
| `REF`, `ALT` | alelo de referencia y alternativo |
| `QUAL`, `FILTER` | calidad de la llamada y si pasó los filtros |
| `INFO`, `FORMAT` | anotaciones de la variante y subcampos por muestra |
| `GT` | el genotipo de cada individuo |

> La explicación de **cada columna** está en [`docs/formato_vcf.md`](docs/formato_vcf.md).


## 4. Los datos del curso: simulados y reales

Usaremos **dos tipos de datos**:

| | Simulados | Reales (subconjunto) |
|---|---|---|
| Origen | `00_simular_datos.R` | Sekely et al. 2024 (Zenodo, CC-BY-4.0) |
| Tamaño | 20 poblaciones × 12 árboles × 1.500 SNP | 3 sitios × 6 árboles × 301 SNP |
| Para qué | Practicar con una «verdad» conocida | Ver cómo se ve un VCF real |

- Los **simulados** imitan un diseño realista (gradiente latitudinal, más deriva en un extremo) e incluyen trampas deliberadas para entrenar el ojo (una muestra mal etiquetada, datos faltantes).
- Los **reales** son un recorte diminuto y limpio para empezar: `datos_ejemplo/lenga_ejemplo.vcf`.

**El diseño de muestreo del estudio real** (útil como referencia metodológica): 20 sitios distribuidos a lo largo de un gradiente latitudinal, la mayoría con un **par de elevación** (alto/bajo). Ese diseño permite separar el efecto del clima (latitud) del efecto local de la altitud.


## 5. Mapa de los 20 sitios

Veamos cómo se distribuyen los sitios del estudio real:


In [ ]:
# ---- Preparación ----
options(repr.plot.width = 7, repr.plot.height = 5, repr.plot.res = 110)
library(ggplot2)

sitios <- read.csv("datos_ejemplo/lenga_sitios.csv", stringsAsFactors = FALSE)

ggplot(sitios, aes(lon, lat, colour = el_class, size = elev)) +
  geom_point(alpha = 0.85) +
  geom_text(aes(label = loc), nudge_y = 0.35, size = 3, colour = "grey30", show.legend = FALSE) +
  scale_colour_viridis_d(option = "D") +
  scale_size_continuous(range = c(2, 6)) +
  labs(title = "20 sitios de muestreo (Sekely et al. 2024)",
       x = "Longitud", y = "Latitud",
       colour = "Clase de elevación", size = "Elevación (m)") +
  coord_fixed(1.3)


Fíjate en el eje de latitud: abarca casi 20° (~2.000 km). Los sitios **1**, **10** y **20** son los tres que usamos en el subconjunto real del curso.


## Resumen: ideas que debes llevarte

1. El curso sigue un flujo reproducible: **importar → QC → diversidad → estructura → ambiente**.
2. Un VCF es el resultado de un **pipeline**: muestreo → secuenciación → mapeo → llamado → filtrado.
3. La dosis (`0/1/2/NA`) convierte genotipos en números con los que se puede calcular.
4. Cada decisión de análisis tiene **consecuencias**; hay que justificarlas y reportarlas.


## Tarea (para la próxima clase)

### 1. Mirar un VCF real (con el archivo en la mano)

Trabaja sobre `datos_ejemplo/lenga_ejemplo.vcf`. Puedes abrirlo en un **editor de texto** (VS Code, Notepad++, TextEdit) o desde **R**, mirando las primeras líneas:

```r
lineas <- readLines("datos_ejemplo/lenga_ejemplo.vcf", n = 25)
lineas[1:5]        # las líneas de metadatos (##)
lineas[grep("^#CHROM", lineas)]   # la cabecera de columnas
```

Anota en tu cuaderno:

1. **Una línea `##`** y qué declara.
2. **La línea `#CHROM`**: ¿cuántas columnas fijas hay antes de las muestras, y **cuántos individuos** hay en total?
3. **Una fila de variante**: su contig (`CHROM`) y su posición (`POS`), sus alelos `REF` y `ALT`, y el **genotipo (`GT`) de un individuo** — ¿es homocigoto, heterocigoto o faltante?

Para contar cuántos SNP trae el archivo:

```r
datos <- readLines("datos_ejemplo/lenga_ejemplo.vcf")
sum(!grepl("^#", datos))     # número de SNP (filas de datos)
```

### 2. Tu hipótesis

Ahora sí: con el archivo ya visto, escribe tu hipótesis. ¿Dónde esperas **más** diversidad genética en la lenga, en el extremo norte o en el extremo sur del gradiente, y por qué? Dos o tres líneas.

> Escríbela **antes** de leer el paper: si lo lees primero, la respuesta de los autores se te vuelve obvia y deja de ser una predicción tuya. La contrastamos más adelante, cuando calculemos diversidad.

### 3. Revisar el paper para presentarlo

Este artículo es la referencia del curso: más adelante vamos a usar su diseño, sus métodos y sus resultados para comparar con lo que hagamos nosotros. Revísalo y **prepárate para presentarlo** en la próxima clase.

> **Sekely, J., Marchelli, P., Arana, V., Dauphin, B., Mattera, M. G. & Pastorino, M. (2024).**
> *Genomic responses to climate: Understanding local adaptation in the Andean tree species Nothofagus pumilio and implications for a changing world.*
> **Plants, People, Planet**. DOI: **10.1002/ppp3.10504** — acceso abierto.
> Datos depositados en Zenodo: DOI **10.5281/zenodo.10018374** (CC-BY-4.0).

En la presentación contesta estas preguntas metodológicas:

| # | Pregunta |
|---|---|
| 1 | ¿Cuál es el **diseño de muestreo**: cuántos sitios, cuántos árboles y cómo eligieron los pares de elevación alto/bajo? ¿Por qué ese diseño permite separar el efecto del clima del efecto de la geografía? |
| 2 | ¿Cuántos **SNP** secuenciaron, **cómo los obtuvieron** (tipo de captura) y **qué filtros** aplicaron? |
| 3 | ¿Qué **métodos** usaron para diversidad, para estructura y para ambiente? Anota el **nombre de cada uno** y qué mide. |
| 4 | ¿Qué **concluyen** sobre adaptación local y qué papel juega la **altitud** frente a la **latitud**? |
| 5 | ¿Qué **limitación** reconocen los propios autores? ¿Se te ocurre alguna otra? |

Y cierra conectando el paper con nuestro curso: **qué de lo que hicieron vamos a usar nosotros y qué no.**